# EMI-BF4 Geometry Optimization Benchmark

This notebook performs a fast geometry sanity check for the starting neutral EMI-BF4 structure used by the wall-collision workflows. It compares the DFT-optimized geometry metrics reported by Laws et al. (2026) against the input structure and, when MACE is installed, against short geometry optimizations with MACE-medium and MACE-POLAR-1.

The optimization is intentionally small: one 24-atom neutral ion pair, isolated in a large nonperiodic box. Outputs are written to `results/geometry_benchmark/`, which is ignored by Git.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from ase.io import read, write
from ase.optimize import BFGS

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
if not (ROOT / "scripts" / "simulation" / "inputs" / "EMIBF4.dat").exists():
    ROOT = Path.cwd().parents[1]

INPUT = ROOT / "scripts" / "simulation" / "inputs" / "EMIBF4.dat"
OUTDIR = ROOT / "results" / "geometry_benchmark"
OUTDIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda"  # set to "cpu" if needed
DTYPE = "float64"
FMAX = 0.03
MAX_STEPS = 200

print(f"Repository root: {ROOT}")
print(f"Input structure: {INPUT}")
print(f"Output folder: {OUTDIR}")

## Reference Data From Laws et al. (2026)

Atom labels follow the table notation: `N1-C2-N3-C4-C5` are the imidazolium ring atoms and `B1-F1-F2-F3-F4` are the tetrafluoroborate atoms. The tracked `EMIBF4.dat` file uses the same ordering for these atoms.

In [ ]:
metric_specs = [
    # Anion bond lengths
    {"group": "Anion bond length", "metric": "B1-F1", "kind": "distance", "atoms": (19, 20), "dft": 1.364, "experiment": 1.376},
    {"group": "Anion bond length", "metric": "B1-F2", "kind": "distance", "atoms": (19, 21), "dft": 1.365, "experiment": 1.386},
    {"group": "Anion bond length", "metric": "B1-F3", "kind": "distance", "atoms": (19, 22), "dft": 1.378, "experiment": 1.391},
    {"group": "Anion bond length", "metric": "B1-F4", "kind": "distance", "atoms": (19, 23), "dft": 1.703, "experiment": 1.399},
    # External anion angles
    {"group": "External anion angle", "metric": "F1-B1-F2", "kind": "angle", "atoms": (20, 19, 21), "dft": 94.0, "experiment": 108.7},
    {"group": "External anion angle", "metric": "F1-B1-F3", "kind": "angle", "atoms": (20, 19, 22), "dft": 96.0, "experiment": 108.8},
    {"group": "External anion angle", "metric": "F1-B1-F4", "kind": "angle", "atoms": (20, 19, 23), "dft": 96.8, "experiment": 109.0},
    {"group": "External anion angle", "metric": "F2-B1-F3", "kind": "angle", "atoms": (21, 19, 22), "dft": 113.9, "experiment": 109.5},
    {"group": "External anion angle", "metric": "F2-B1-F4", "kind": "angle", "atoms": (21, 19, 23), "dft": 120.0, "experiment": 109.7},
    {"group": "External anion angle", "metric": "F3-B1-F4", "kind": "angle", "atoms": (22, 19, 23), "dft": 123.3, "experiment": 111.1},
    # Cation bond lengths
    {"group": "Cation bond length", "metric": "N1-C2", "kind": "distance", "atoms": (0, 1), "dft": 1.292, "experiment": 1.330},
    {"group": "Cation bond length", "metric": "C2-N3", "kind": "distance", "atoms": (1, 2), "dft": 1.305, "experiment": 1.335},
    {"group": "Cation bond length", "metric": "N3-C4", "kind": "distance", "atoms": (2, 3), "dft": 1.346, "experiment": 1.361},
    {"group": "Cation bond length", "metric": "C4-C5", "kind": "distance", "atoms": (3, 4), "dft": 1.436, "experiment": 1.384},
    {"group": "Cation bond length", "metric": "C5-N1", "kind": "distance", "atoms": (4, 0), "dft": 1.452, "experiment": 1.390},
    # Internal cation angles
    {"group": "Internal cation angle", "metric": "C5-N1-C2", "kind": "angle", "atoms": (4, 0, 1), "dft": 104.8, "experiment": 106.3},
    {"group": "Internal cation angle", "metric": "N1-C2-N3", "kind": "angle", "atoms": (0, 1, 2), "dft": 106.5, "experiment": 107.8},
    {"group": "Internal cation angle", "metric": "C2-N3-C4", "kind": "angle", "atoms": (1, 2, 3), "dft": 109.2, "experiment": 107.8},
    {"group": "Internal cation angle", "metric": "N3-C4-C5", "kind": "angle", "atoms": (2, 3, 4), "dft": 109.3, "experiment": 108.9},
    {"group": "Internal cation angle", "metric": "C4-C5-N1", "kind": "angle", "atoms": (3, 4, 0), "dft": 110.0, "experiment": 109.3},
]

reference = pd.DataFrame(metric_specs)[["group", "metric", "kind", "dft", "experiment"]]
reference

## Load Starting Geometry

In [ ]:
def prepare_isolated(atoms, cell_A=35.0):
    atoms = atoms.copy()
    atoms.set_cell(np.eye(3) * cell_A)
    atoms.center()
    atoms.set_pbc(False)
    return atoms

start = read(INPUT, format="lammps-data", atom_style="full")
start = prepare_isolated(start)
print(len(start), start.get_chemical_formula())
for i, symbol in enumerate(start.get_chemical_symbols()):
    print(f"{i:2d} {symbol}")

write(OUTDIR / "EMIBF4_start.xyz", start)

In [ ]:
def metric_value(atoms, spec):
    if spec["kind"] == "distance":
        i, j = spec["atoms"]
        return float(atoms.get_distance(i, j, mic=False))
    if spec["kind"] == "angle":
        i, j, k = spec["atoms"]
        return float(atoms.get_angle(i, j, k, mic=False))
    raise ValueError(spec["kind"])

def measure_geometry(atoms, label):
    rows = []
    for spec in metric_specs:
        rows.append({
            "method": label,
            "group": spec["group"],
            "metric": spec["metric"],
            "kind": spec["kind"],
            "value": metric_value(atoms, spec),
            "dft": spec["dft"],
            "experiment": spec["experiment"],
        })
    return pd.DataFrame(rows)

input_metrics = measure_geometry(start, "Input")
input_metrics.head()

## MACE Calculators

The cells below are written to be robust across MACE versions. If MACE is not installed in the current environment, the notebook will skip the optimizations and still report the reference/input geometry table.

In [ ]:
def _call_with_fallback(factory, **kwargs):
    try:
        return factory(**kwargs)
    except TypeError:
        reduced = {k: v for k, v in kwargs.items() if k not in {"default_dtype"}}
        return factory(**reduced)

def build_mace_medium(device=DEVICE, dtype=DTYPE):
    from mace.calculators import mace_mp
    return _call_with_fallback(mace_mp, model="medium", device=device, default_dtype=dtype)

def build_mace_polar(device=DEVICE, dtype=DTYPE):
    from mace.calculators import mace_polar
    return _call_with_fallback(mace_polar, model="polar-1-m", device=device, default_dtype=dtype)

def optimize_geometry(label, calc_factory):
    atoms = start.copy()
    atoms.calc = calc_factory()
    trajectory = OUTDIR / f"{label.replace(' ', '_').replace('-', '_')}.traj"
    logfile = OUTDIR / f"{label.replace(' ', '_').replace('-', '_')}.log"
    opt = BFGS(atoms, trajectory=str(trajectory), logfile=str(logfile))
    opt.run(fmax=FMAX, steps=MAX_STEPS)
    write(OUTDIR / f"{label.replace(' ', '_').replace('-', '_')}_optimized.xyz", atoms)
    return atoms

optimized = {}
errors = {}
for label, factory in [("MACE-medium", build_mace_medium), ("MACE-POLAR-1", build_mace_polar)]:
    try:
        print(f"Optimizing with {label}...")
        optimized[label] = optimize_geometry(label, factory)
    except Exception as exc:
        errors[label] = repr(exc)
        print(f"Skipped {label}: {exc!r}")

errors

## Geometry Comparison Table

In [ ]:
frames = [input_metrics]
for label, atoms in optimized.items():
    frames.append(measure_geometry(atoms, label))

long_table = pd.concat(frames, ignore_index=True)
wide = long_table.pivot_table(
    index=["group", "metric", "kind", "dft", "experiment"],
    columns="method",
    values="value",
).reset_index()

method_columns = [c for c in ["Input", "MACE-medium", "MACE-POLAR-1"] if c in wide.columns]
for method in method_columns:
    wide[f"{method} abs. error vs DFT"] = (wide[method] - wide["dft"]).abs()

wide.to_csv(OUTDIR / "geometry_metric_comparison.csv", index=False)
long_table.to_csv(OUTDIR / "geometry_metric_comparison_long.csv", index=False)
with (OUTDIR / "optimization_errors.json").open("w", encoding="utf-8") as handle:
    json.dump(errors, handle, indent=2)

display_columns = ["group", "metric", "kind", "dft", "experiment"] + method_columns + [f"{m} abs. error vs DFT" for m in method_columns]
wide[display_columns].round(4)

In [ ]:
summary_rows = []
for method in method_columns:
    subset = wide[["kind", f"{method} abs. error vs DFT"]].copy()
    for kind, unit in [("distance", "A"), ("angle", "deg")]:
        values = subset.loc[subset["kind"] == kind, f"{method} abs. error vs DFT"]
        summary_rows.append({
            "method": method,
            "metric_type": kind,
            "unit": unit,
            "mean_abs_error_vs_DFT": values.mean(),
            "max_abs_error_vs_DFT": values.max(),
        })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTDIR / "geometry_error_summary.csv", index=False)
summary.round(4)

## Notes For Interpretation

- The Laws et al. DFT table contains an elongated `B1-F4` DFT distance relative to crystallography. That metric should be inspected carefully because it strongly affects anion bond-length error summaries.
- MACE-medium and MACE-POLAR-1 are foundation models, not refit potentials for this specific neutral EMI-BF4 geometry. This notebook is a fast geometry benchmark, not a replacement for DFT validation.
- If `MACE-POLAR-1` fails because the package or model is unavailable, rerun in the same environment used for the wall-collision simulations.